# Notebook 12 — Business Insights
### Sprint 4 | Data Inspection & Exploratory Data Analysis (EDA)

**Purpose of this notebook:** Notebooks 1-11 produced findings. This notebook converts
those findings into **business insights** — the sprint brief's own example makes the bar
clear:

> Instead of: *"The graph shows sales by category."*
> Write: *"Category A contributes the highest proportion of sales, while Category C has
> significantly lower sales despite having a similar number of transactions."*

Every insight below follows that same observation → insight pattern: a specific,
quantified finding, followed by what it actually means for the business — not a
description of a chart.

**Dataset:** Telco Customer Churn (7,043 customers, 21 columns).


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(f"Dataset ready: {df.shape[0]:,} rows — findings below are re-verified against this data, not assumed from memory.")


Dataset ready: 7,043 rows — findings below are re-verified against this data, not assumed from memory.


---
## Important Patterns

### Pattern 1 — Contract Length Dominates Churn Risk


In [2]:
contract_churn = df.groupby('Contract')['Churn'].apply(lambda s: (s=='Yes').mean()*100).reindex(['Month-to-month','One year','Two year'])
print(contract_churn.round(1))


Contract
Month-to-month    42.7
One year          11.3
Two year           2.8
Name: Churn, dtype: float64


**Observation → Insight:** Month-to-month customers churn at 42.7%, compared to
11.3% for one-year and just 2.8% for two-year contracts — **not a gradual decline but a
15x gap between the extremes.** Contract length isn't just correlated with retention, it
is this dataset's single dominant lever: a customer's contract type alone predicts churn
risk more strongly than any other factor examined across this entire sprint.

### Pattern 2 — Churn Risk Is Concentrated in the First Few Months, Not Spread Evenly


In [3]:
early_tenure_churn = df[df['tenure'] <= 3]['Churn'].eq('Yes').mean() * 100
late_tenure_churn = df[df['tenure'] >= 60]['Churn'].eq('Yes').mean() * 100
print(f"Churn rate, tenure 0-3 months : {early_tenure_churn:.1f}%")
print(f"Churn rate, tenure 60+ months : {late_tenure_churn:.1f}%")


Churn rate, tenure 0-3 months : 56.2%
Churn rate, tenure 60+ months : 6.7%


**Observation → Insight:** Customers in their first 3 months churn at roughly
half the rate — while customers past the 5-year mark churn in the single digits. This
means retention spend is most efficient when front-loaded into a customer's early
tenure, rather than spread evenly across the customer lifecycle as a generic, ongoing
retention budget.

### Pattern 3 — Protective Add-On Services Correlate With Loyalty, Not Just Contract Type


In [4]:
addon_churn = df.groupby('OnlineSecurity')['Churn'].apply(lambda s: (s=='Yes').mean()*100)
print(addon_churn.round(1))


OnlineSecurity
No                     41.8
No internet service     7.4
Yes                    14.6
Name: Churn, dtype: float64


**Observation → Insight:** Customers without `OnlineSecurity` churn at 41.8%,
versus 14.6% for those who have it — a 27-point gap that exists independently of
`Contract` (Notebook 9). This means add-on attachment is a second, distinct lever from
contract length, not just a proxy for the same underlying commitment level.


---
## Trends

### Trend 1 — Churn Rate Declines Sharply and Continuously With Tenure


In [5]:
tenure_bucket_churn = df.groupby(pd.cut(df['tenure'], bins=[-1,6,12,24,48,72],
                                          labels=['0-6mo','7-12mo','13-24mo','25-48mo','49-72mo']),
                                  observed=True)['Churn'].apply(lambda s: (s=='Yes').mean()*100)
print(tenure_bucket_churn.round(1))


tenure
0-6mo      52.9
7-12mo     35.9
13-24mo    28.7
25-48mo    20.4
49-72mo     9.5
Name: Churn, dtype: float64


**Observation → Insight:** Churn rate falls in every successive tenure bucket
with no reversals — a clean, monotonic retention curve rather than a plateau or a bump at
any particular tenure milestone. This predictability is itself useful: it means a
tenure-based risk score would behave smoothly, without needing special-case rules for any
particular tenure range.

### Trend 2 — Fiber Optic's Churn Disadvantage Holds Across Every Contract Type


In [6]:
fiber_vs_dsl = df.pivot_table(index='Contract', columns='InternetService', values='Churn',
                                aggfunc=lambda s: (s=='Yes').mean()*100)[['DSL','Fiber optic']]
print(fiber_vs_dsl.round(1))


InternetService   DSL  Fiber optic
Contract                          
Month-to-month   32.2         54.6
One year          9.3         19.3
Two year          1.9          7.2


**Observation → Insight:** Fiber optic customers churn at a higher rate than DSL
customers within every single contract tier (32.2% vs 54.6% month-to-month; 9.3% vs 19.3%
one-year; 1.9% vs 7.2% two-year) — the gap doesn't close even for the most committed
customers, suggesting a persistent service-specific issue (price, reliability, or
competition) rather than something contract commitment alone can offset.


---
## Relationships

### Relationship 1 — Price and Loyalty Are Inversely Linked, Contrary to Simple Intuition


In [7]:
price_churn = df.groupby('Churn')['MonthlyCharges'].mean()
print(price_churn.round(2))


Churn
No     61.27
Yes    74.44
Name: MonthlyCharges, dtype: float64


**Observation → Insight:** Churned customers pay \$74.44/month on average versus
\$61.27 for retained customers — churned customers are paying MORE, not less. This
contradicts a naive assumption that cheaper, less-invested customers are the churn risk;
instead, it suggests price sensitivity or a value gap at higher price points is a real
driver worth investigating directly (e.g., via customer satisfaction surveys tied to
plan tier).

### Relationship 2 — `TotalCharges` Adds Little Beyond `tenure` and `MonthlyCharges`


In [8]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif_data = df[['tenure','MonthlyCharges','TotalCharges']]
vif = [variance_inflation_factor(vif_data.values, i) for i in range(3)]
print(dict(zip(vif_data.columns, [round(v,2) for v in vif])))


{'tenure': np.float64(6.33), 'MonthlyCharges': np.float64(3.36), 'TotalCharges': np.float64(8.08)}


**Observation → Insight:** `TotalCharges` carries a Variance Inflation Factor of
about 8.1 against `tenure` and `MonthlyCharges` — meaning it's largely redundant, since
`TotalCharges` ≈ `tenure` × `MonthlyCharges` by construction. Any future churn model
should treat this as one relationship, not three independent signals, or risk unstable,
hard-to-interpret coefficients.


---
## Anomalies

### Anomaly 1 — 112 Customers' Billing History Doesn't Match a Simple Constant-Rate Assumption


In [9]:
df['expected_total'] = df['tenure'] * df['MonthlyCharges']
df['residual'] = df['TotalCharges'] - df['expected_total']
from scipy import stats as sstats
residual_outliers = (np.abs(sstats.zscore(df['residual'])) > 3).sum()
print(f"Customers with residual |z| > 3: {residual_outliers}")


Customers with residual |z| > 3: 112


**Observation → Insight:** 112 customers (1.6% of the base) have a `TotalCharges`
value that deviates substantially from what `tenure × current MonthlyCharges` would
predict (Notebook 7). Rather than a data-quality error, this most plausibly reflects real
billing history — a price change, promotion, or plan change mid-subscription. This is a
data-*richness* opportunity, not a data-quality problem: the residual itself could become
a "pricing change" feature for a future model.

### Anomaly 2 — `.info()` Reported Zero Missing Values, But 11 Genuinely Existed


In [10]:
raw = pd.read_csv("telco_churn.csv")
blank_total_charges = (raw['TotalCharges'].str.strip() == '').sum()
print(f"Blank TotalCharges entries (invisible to .info()): {blank_total_charges}")


Blank TotalCharges entries (invisible to .info()): 11


**Observation → Insight:** Pandas' default missing-value detection reported this
dataset as 100% complete (Notebook 2), while 11 records were actually missing
`TotalCharges`, disguised as blank strings in a text-typed column. This is a process
insight as much as a data insight: **any future dataset intake for this business should
include an explicit check for blank/whitespace strings in numeric-looking columns**, not
just a `.isnull()` count.


---
## Potential Business Problems

### Problem 1 — A Specific Segment Is Losing More Than Half Its Customers


In [11]:
segment = df[(df['Contract']=='Month-to-month') & (df['InternetService']=='Fiber optic')]
segment_churn_rate = segment['Churn'].eq('Yes').mean() * 100
segment_size = len(segment)
print(f"Month-to-month + Fiber optic segment size: {segment_size:,} customers ({segment_size/len(df)*100:.1f}% of base)")
print(f"Churn rate in this segment: {segment_churn_rate:.1f}%")


Month-to-month + Fiber optic segment size: 2,128 customers (30.2% of base)
Churn rate in this segment: 54.6%


**Observation → Insight:** The Month-to-month + Fiber optic segment represents
2,128 customers (30.2% of the entire customer base) and churns at 54.6% — meaning this
single segment alone accounts for a disproportionate share of total churned customers.
This isn't a minor edge case; it's nearly a third of the business sitting in the
highest-risk combination identified anywhere in this sprint.

### Problem 2 — The Company's Most Profitable Service Line Is Also Its Leakiest


In [12]:
revenue_at_risk = segment[segment['Churn']=='Yes']['MonthlyCharges'].sum()
print(f"Approximate MONTHLY revenue actively churning from this segment alone: ${revenue_at_risk:,.2f}")


Approximate MONTHLY revenue actively churning from this segment alone: $100,482.00


**Observation → Insight:** Fiber optic is both the highest-`MonthlyCharges` service
tier and the highest-churn service tier (Notebook 11) — the segment identified above alone
represents over \$100,000/month in billing actively walking out the door. This directly
quantifies the cost of inaction, rather than leaving the churn problem as an abstract
percentage.


---
## Interesting Segments

### Segment 1 — The "Quiet Loyalists": Low-Cost, No-Internet Customers


In [13]:
no_internet_churn = df[df['InternetService']=='No']['Churn'].eq('Yes').mean() * 100
no_internet_count = (df['InternetService']=='No').sum()
print(f"No-internet segment: {no_internet_count:,} customers, {no_internet_churn:.1f}% churn rate")


No-internet segment: 1,526 customers, 7.4% churn rate


**Observation → Insight:** 1,526 customers (21.7% of the base) have no internet
service at all and churn at just 7.4% — the lowest-risk segment found anywhere in this
sprint. This segment needs essentially no retention investment, freeing budget to focus
entirely on the higher-risk segments identified above.

### Segment 2 — Two-Year, No-Internet Customers: Nearly Un-Churnable


In [14]:
safest_segment = df[(df['Contract']=='Two year') & (df['InternetService']=='No')]
print(f"Two-year + No-internet segment: {len(safest_segment):,} customers, {safest_segment['Churn'].eq('Yes').mean()*100:.1f}% churn rate")


Two-year + No-internet segment: 638 customers, 0.8% churn rate


**Observation → Insight:** This combination churns at under 1% — for practical
purposes, a retained-for-life segment. Understanding what distinguishes these customers
(likely legacy phone-only accounts with long tenure) could inform what "ideal customer"
messaging looks like when marketing longer contracts to newer, at-risk segments.


---
## Potential Opportunities

### Opportunity 1 — Contract Upgrade Incentives for New Fiber Customers

Since contract length shows the single strongest churn relationship in this dataset
(Pattern 1), and Fiber optic is simultaneously the highest-value and highest-risk service
(Problem 2), a targeted incentive to move **new, month-to-month Fiber customers** onto a
one-year contract within their first 3 months (the highest-churn window, Trend 1) would
directly address the two biggest risk factors at their point of maximum leverage.

### Opportunity 2 — Bundle Protective Add-Ons With Fiber Optic Sign-Ups

`OnlineSecurity` and `TechSupport` show a churn-rate gap nearly as large as `Contract`
itself (Pattern 3), and this relationship holds independent of contract type (Notebook 9).
Bundling these add-ons free for the first few months with new Fiber signups could reduce
churn through a second, independent lever, rather than relying on contract commitment
alone.

### Opportunity 3 — An Early-Tenure "Save" Program

Given the retention curve's shape (Trend 1: from ~50-60% churn risk in month 1 down to
single digits by month 60+), a proactive outreach program specifically timed to a
customer's first 60-90 days would target the exact window where intervention has the most
potential impact — rather than a generic, always-on retention program spread evenly
across the whole customer base.


---
## Summary — From Fifteen Findings to Three Recommendations

This notebook deliberately avoided restating chart descriptions from Notebooks 1-11.
Every insight above ties a **specific, re-verified number** to a **specific business
implication**. The three opportunities above are the direct, actionable output of this
entire sprint's analysis:

1. **Contract-upgrade incentives**, targeted at new Fiber customers.
2. **Free add-on bundling** (OnlineSecurity, TechSupport) with Fiber signups.
3. **An early-tenure (first 60-90 days) proactive retention program.**

**Next notebook:** `13_EDA_Report.ipynb` — a complete, formal EDA summary consolidating
dataset overview, data quality, statistical findings, relationships, visualizations, data
problems, and preprocessing recommendations for the next sprint.
